In [8]:
import pandas as pd

decisao = pd.read_csv("../data/eventos_com_decisao.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

spy = pd.read_csv("../data/spy_precos.csv")
ewz = pd.read_csv("../data/ewz_precos.csv")
usdbrl = pd.read_csv("../data/usdbrl_precos.csv")

spy["ativo"] = "SPY"
ewz["ativo"] = "EWZ"
usdbrl["ativo"] = "BRL=X"

precos_todos = pd.concat([spy, ewz, usdbrl], ignore_index=True)

print(f"Total decisao: {len(decisao)}")
print(decisao["indicador"].value_counts())

Total decisao: 134
indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [9]:
decisao_completa = decisao.merge(playbooks[["indicador", "ativo_alvo"]], on="indicador", how="left")
decisao_completa = decisao_completa.merge(
    precos_todos[["data", "ativo", "retorno_pct"]],
    left_on=["data", "ativo_alvo"],
    right_on=["data", "ativo"],
    how="left"
)

operacoes = decisao_completa[decisao_completa["opera"]].copy()
print(operacoes[["data", "indicador", "ativo_alvo", "direcao", "retorno_pct"]])
print(f"\nLinhas com retorno faltando: {operacoes['retorno_pct'].isna().sum()}")

           data    indicador ativo_alvo  direcao  retorno_pct
1    2023-04-12      CPI_EUA        SPY  comprar    -0.407599
13   2024-04-10      CPI_EUA        SPY   vender    -1.001323
33   2026-02-13      CPI_EUA        SPY  comprar     0.070450
35   2026-04-10      CPI_EUA        SPY  comprar    -0.066178
38   2026-07-14      CPI_EUA        SPY  comprar     0.355064
52   2024-02-01      IPCA_BR        EWZ   vender     1.123939
63   2025-01-01      IPCA_BR        EWZ  comprar          NaN
64   2025-02-01      IPCA_BR        EWZ   vender          NaN
65   2025-03-01      IPCA_BR        EWZ   vender          NaN
73   2023-08-03     Selic_BR      BRL=X  comprar     0.383977
82   2024-12-12     Selic_BR      BRL=X   vender    -1.531766
128  2026-03-06  Payroll_EUA        SPY  comprar    -1.310713
129  2026-04-03  Payroll_EUA        SPY   vender          NaN
131  2026-06-05  Payroll_EUA        SPY   vender    -2.580937

Linhas com retorno faltando: 4


In [10]:
def calcular_retorno_operacao(row):
    if pd.isna(row["retorno_pct"]):
        return None
    if row["direcao"] == "comprar":
        return row["retorno_pct"] * row["tamanho_posicao"]
    elif row["direcao"] == "vender":
        return -row["retorno_pct"] * row["tamanho_posicao"]
    return 0

operacoes["retorno_operacao"] = operacoes.apply(calcular_retorno_operacao, axis=1)
operacoes = operacoes.dropna(subset=["retorno_operacao"])
print(operacoes[["data", "indicador", "direcao", "retorno_pct", "tamanho_posicao", "retorno_operacao"]])

           data    indicador  direcao  retorno_pct  tamanho_posicao  \
1    2023-04-12      CPI_EUA  comprar    -0.407599         0.406269   
13   2024-04-10      CPI_EUA   vender    -1.001323         0.270855   
33   2026-02-13      CPI_EUA  comprar     0.070450         0.378187   
35   2026-04-10      CPI_EUA  comprar    -0.066178         0.292912   
38   2026-07-14      CPI_EUA  comprar     0.355064         0.287964   
52   2024-02-01      IPCA_BR   vender     1.123939         0.532847   
73   2023-08-03     Selic_BR  comprar     0.383977         0.933219   
82   2024-12-12     Selic_BR   vender    -1.531766         0.461880   
128  2026-03-06  Payroll_EUA  comprar    -1.310713         0.713499   
131  2026-06-05  Payroll_EUA   vender    -2.580937         0.963315   

     retorno_operacao  
1           -0.165595  
13           0.271213  
33           0.026643  
35          -0.019384  
38           0.102246  
52          -0.598888  
73           0.358335  
82           0.707492  
12

In [11]:
capital_inicial = 100
operacoes = operacoes.sort_values("data")
operacoes["capital"] = capital_inicial * (1 + operacoes["retorno_operacao"] / 100).cumprod()
print(operacoes[["data", "indicador", "retorno_operacao", "capital"]])

           data    indicador  retorno_operacao     capital
1    2023-04-12      CPI_EUA         -0.165595   99.834405
73   2023-08-03     Selic_BR          0.358335  100.192147
52   2024-02-01      IPCA_BR         -0.598888   99.592108
13   2024-04-10      CPI_EUA          0.271213   99.862215
82   2024-12-12     Selic_BR          0.707492  100.568732
33   2026-02-13      CPI_EUA          0.026643  100.595527
128  2026-03-06  Payroll_EUA         -0.935192   99.654765
35   2026-04-10      CPI_EUA         -0.019384   99.635448
131  2026-06-05  Payroll_EUA          2.486255  102.112639
38   2026-07-14      CPI_EUA          0.102246  102.217045


In [12]:
retorno_total = (operacoes["capital"].iloc[-1] / capital_inicial - 1) * 100
volatilidade = operacoes["retorno_operacao"].std()
sharpe = operacoes["retorno_operacao"].mean() / operacoes["retorno_operacao"].std()
win_rate = (operacoes["retorno_operacao"] > 0).mean() * 100

pico = operacoes["capital"].cummax()
drawdown = (operacoes["capital"] - pico) / pico
max_drawdown = drawdown.min() * 100

print(f"Retorno total: {retorno_total:.2f}%")
print(f"Volatilidade: {volatilidade:.2f}%")
print(f"Sharpe: {sharpe:.2f}")
print(f"Máx. drawdown: {max_drawdown:.2f}%")
print(f"Win rate: {win_rate:.2f}%")

Retorno total: 2.22%
Volatilidade: 0.92%
Sharpe: 0.24
Máx. drawdown: -0.95%
Win rate: 60.00%


In [16]:
print(metricas_por_indicador)

             total_operacoes  retorno_medio  win_rate
indicador                                            
CPI_EUA                    5           0.04      60.0
IPCA_BR                    1          -0.60       0.0
Payroll_EUA                2           0.78      50.0
Selic_BR                   2           0.53     100.0


In [13]:
metricas_por_indicador = operacoes.groupby("indicador").agg(
    total_operacoes=("retorno_operacao", "count"),
    retorno_medio=("retorno_operacao", "mean"),
    win_rate=("retorno_operacao", lambda x: (x > 0).mean() * 100)
).round(2)
print(metricas_por_indicador)

             total_operacoes  retorno_medio  win_rate
indicador                                            
CPI_EUA                    5           0.04      60.0
IPCA_BR                    1          -0.60       0.0
Payroll_EUA                2           0.78      50.0
Selic_BR                   2           0.53     100.0


In [14]:
operacoes.to_csv("../data/resultado_backtest.csv", index=False)